In [1]:
import pandas as pd

soc1018 = pd.read_excel("../data/raw/bls/soc_2010_to_2018_crosswalk.xlsx",
                        sheet_name="Sorted by 2010", header=8, dtype=str)
isco = pd.read_excel("../data/raw/bls/ISCO_SOC_Crosswalk.xls",
                     sheet_name="2010 SOC to ISCO-08", header=6, dtype=str)

C:\Users\ferra\AppData\Local\Temp\ipykernel_15256\744695711.py:5: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  isco = pd.read_excel("../data/raw/bls/ISCO_SOC_Crosswalk.xls",


In [2]:

d1 = soc1018[["2018 SOC Code", "2010 SOC Code"]].apply(lambda col: col.str.strip()).drop_duplicates()
d1.columns = ["soc2018", "soc2010"]
d1["w1"] = 1 / d1.groupby("soc2018")["soc2010"].transform("count")
d2 = isco[["2010 SOC Code", "ISCO-08 Code"]].apply(lambda col: col.str.strip())
d2.columns = ["soc2010", "isco08"]
d2["isco2"] = "OC" + d2["isco08"].str[:2]
d2 = d2[["soc2010", "isco2"]].drop_duplicates()
d2["w2"] = 1 / d2.groupby("soc2010")["isco2"].transform("count")

In [3]:
chain = d1.merge(d2, on="soc2010", how="left")
chain["w"] = chain["w1"] * chain["w2"]
xw = chain.groupby(["soc2018", "isco2"], as_index=False)["w"].sum()
print(xw[xw["soc2018"] == "15-1299"])
print(xw.groupby("soc2018")["w"].sum().round(6).value_counts())

     soc2018 isco2    w
116  15-1299  OC25  0.5
117  15-1299  OC35  0.5
w
1.0    867
Name: count, dtype: int64


In [4]:
df = pd.read_csv("../data/raw/anthropic/aei_claude_ai_2026-06-26.csv",
                 dtype={"node_external_id": str}, low_memory=False)
jobs = df[(df["geo_level"] == "country") & (df["category_name"] == "soc_occupation") &
          (df["hierarchy_level"] == 0) & (df["metric_id"] == "pct")].copy()
jobs["soc2018"] = jobs["node_external_id"].str[:7]
m = jobs.merge(xw, on="soc2018", how="left")
m["pct_isco"] = m["value"] * m["w"]

print("Codes with no match:", m["w"].isna().sum())
before = jobs.groupby(["geo_id", "date_start"])["value"].sum()
after = m.groupby(["geo_id", "date_start"])["pct_isco"].sum()
print("Largest difference:", (before - after).abs().max().round(6))

Codes with no match: 0
Largest difference: 0.0


In [5]:
usage = m.groupby(["geo_id", "date_start", "isco2"], as_index=False)["pct_isco"].sum()
usage.to_csv("../data/processed/usage_shares.csv", index=False)

In [6]:
us = pd.read_excel("../data/raw/bls/national_M2025_dl.xlsx", dtype={"OCC_CODE": str})
us = us[us["O_GROUP"] == "detailed"][["OCC_CODE", "TOT_EMP"]]
us.columns = ["soc2018", "us_emp"]
us["us_emp"] = pd.to_numeric(us["us_emp"], errors="coerce")

size = xw.merge(us, on="soc2018", how="inner")
size["us_emp_isco"] = size["us_emp"] * size["w"]
group_size = size.groupby("isco2", as_index=False)["us_emp_isco"].sum()
group_size.columns = ["isco2", "us_size"]
print(len(us), len(group_size))

830 41


In [7]:
xw_us = xw.merge(group_size, on="isco2", how="left")
xw_us["w_us"] = xw_us["us_size"] / xw_us.groupby("soc2018")["us_size"].transform("sum")
print(xw_us.groupby("soc2018")["w_us"].sum().round(6).value_counts())
xw_us[xw_us["soc2018"] == "15-1299"]

w_us
1.0    856
0.0     11
Name: count, dtype: int64


,soc2018,isco2,w,us_size,w_us
116,15-1299,OC25,0.5,3.881614e+06,0.723091
117,15-1299,OC35,0.5,1.486472e+06,0.276909


In [8]:
m_us = jobs.merge(xw_us[["soc2018", "isco2", "w_us"]], on="soc2018", how="left")
m_us["pct_isco"] = m_us["value"] * m_us["w_us"]
before = jobs.groupby(["geo_id", "date_start"])["value"].sum()
after = m_us.groupby(["geo_id", "date_start"])["pct_isco"].sum()
print((before - after).abs().max())

usage_us = m_us.groupby(["geo_id", "date_start", "isco2"], as_index=False)["pct_isco"].sum()
usage_us.to_csv("../data/processed/usage_shares_usw.csv", index=False)

1.4210854715202004e-14


In [9]:
ex = pd.read_csv("../data/raw/exposure/occ_level.csv")
ex["soc2018"] = ex["O*NET-SOC Code"].str[:7]
ex = ex.groupby("soc2018", as_index=False)[["human_rating_beta", "dv_rating_beta"]].mean()

x = xw.merge(ex, on="soc2018", how="inner").merge(us, on="soc2018", how="inner")
x["wt"] = x["w"] * x["us_emp"]
x["h_wt"] = x["human_rating_beta"] * x["wt"]
x["g_wt"] = x["dv_rating_beta"] * x["wt"]
expo = x.groupby("isco2", as_index=False)[["h_wt", "g_wt", "wt"]].sum()
expo["X_human"] = expo["h_wt"] / expo["wt"]
expo["X_gpt"] = expo["g_wt"] / expo["wt"]
expo = expo[["isco2", "X_human", "X_gpt"]]
expo.to_csv("../data/processed/exposure_isco2.csv", index=False)
print(len(ex), len(expo))
expo.sort_values("X_human").round(2)

798 41


,isco2,X_human,X_gpt
38,OC94,0.02,0.01
36,OC92,0.03,0.04
33,OC82,0.03,0.09
35,OC91,0.04,0.01
32,OC81,0.06,0.07
26,OC63,0.07,0.13
28,OC72,0.08,0.12
37,OC93,0.09,0.09
27,OC71,0.09,0.12
40,OC96,0.10,0.11
